# **Objective - Customer Segmentation**

Customer segmentation is a crucial technique for any business.  
It helps in understanding customers based on their purchase behavior, demographics, and other characteristics, ultimately leading to informed marketing decisions.

Effective customer segmentation allows businesses to allocate marketing resources efficiently.  
For instance, after segmenting customers, a business might identify a group of loyal customers who significantly contribute to the business's success.  
Conversely, some customers may show signs of churning. By understanding these segments, a business can tailor its strategies to maximize profits and minimize losses.

In this notebook, I demonstrate how customer segmentation, based on purchase behavior, is done on a UK-based and registered non-store online retail dataset in the following process:  
1. Data Exploration
2. Data Cleaning
3. Feature Engineering
4. Data Preprocessing
5. KMeans Clustering
6. Share Results
7. Future Work

In [ ]:
# The code below ignores warnings generated from certain cells
import warnings
warnings.filterwarnings('ignore')

# **1-Data Exploration**

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('/kaggle/input/ecommerce-data/data.csv', encoding='ISO-8859-1')

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,6))

sns.kdeplot(data=df, x='Quantity', ax=axes[0])
axes[0].set_title('Distribution of Quantity')

sns.kdeplot(data=df, x='UnitPrice', ax=axes[1])
axes[1].set_title('Distribution of Unit Price')

plt.tight_layout()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df[['CustomerID','Description','Country']].nunique()

In [ ]:
df['Country'].value_counts().head()

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"First invoice recorded in: {df['InvoiceDate'].min()}")
print(f"Last invoice recorded in: {df['InvoiceDate'].max()}")

Based on the results above, between Decemeber 1st, 2010 and December 9th, 2011, we notice that:  

* There are 4372 unique customers recorded in that time period.  
* 4223 unique products sold.  
* Orders were made from 38 different countries, and the UK has the highest share of orders.  
* 135080 invoices with no customers IDs recorded.  
* 1454 invoices with no product's description.  
* The distribution of quantity column spans from extreme negative values to extreme positive values.  
* The distribution of unit price column, also, spans from extreme negative values to extreme positive values.  
* There are 5268 duplicate records.

# **2-Data Cleaning**


Since I'm going to perform customer segmentation, specifically based on purchase behavior, I'll only need the following columns:  
* Invoice number  
* Invoice date  
* Customer ID  
* Unit price  
* Quantity  

Also, for data cleaning, I'll go over the following steps along with the logic behind each:  

a) Removing duplicates. They were most likely recorded due to a system error.  
b) Subsetting columns of interest for easier preprocessing.  
c) Removing invoices with null customer IDs since the analysis requires known IDs.  
d) Changing customer ID column type to 'object' to perform grouping.  
e) Removing invoices with negative quantity and unit price values, as well as zero values if applicable.  
Negative values refer to cancelled invoices.  
f) Removing outliers in both quantity and unit price columns. Having outliers usually distort the results of most analyses.  
I'll use IQR method to remove them since I don't have access to the business to help me decide how to deal with outliers.

## a) Duplicates

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

## b) Subset

In [ ]:
df_filtered = df[['InvoiceNo', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID']].copy()

## c) Null Values

In [ ]:
df_filtered.isnull().sum()

In [ ]:
df_filtered = df_filtered.dropna()
df_filtered.isnull().sum()

## 4) Data Types

In [ ]:
df_filtered['CustomerID'] = df_filtered['CustomerID'].round(0).astype('str')

In [ ]:
df_filtered.dtypes

## 5) Negative Quantity and Unit Price Entries

In [ ]:
positive_quantity = df_filtered['Quantity'] > 0
positive_unitprice = df_filtered['UnitPrice'] > 0

df_filtered = df_filtered[positive_quantity & positive_unitprice]

fig, axes = plt.subplots(1,2, figsize=(14,6))

sns.kdeplot(data=df_filtered, x='Quantity', ax=axes[0])
axes[0].set_title('Distribution of Quantity')

sns.kdeplot(data=df_filtered, x='UnitPrice', ax=axes[1])
axes[1].set_title('Distribution of Unit Price')

plt.tight_layout()

## 6) Outliers

In [ ]:
# The factor determines how many points to clip (the higher the value, the less points to clip and vise versa)
factor = 2

def calculate_limits(series, factor):
    q1 = np.quantile(series, 0.25)
    q3 = np.quantile(series, 0.75)
    iqr = q3 - q1
    lower_limit = q1 - iqr * factor
    upper_limit = q3 + iqr * factor
    return lower_limit, upper_limit

# Calculate limits for Quantity
lower_limit_quantity, upper_limit_quantity = calculate_limits(df_filtered['Quantity'], factor)

# Calculate limits for UnitPrice
lower_limit_unitprice, upper_limit_unitprice = calculate_limits(df_filtered['UnitPrice'], factor)

# Filter and remove outliers
df_filtered = df_filtered[
    (df_filtered['Quantity'] > lower_limit_quantity) &
    (df_filtered['Quantity'] < upper_limit_quantity) &
    (df_filtered['UnitPrice'] > lower_limit_unitprice) &
    (df_filtered['UnitPrice'] < upper_limit_unitprice)]

In [ ]:
# Show distributions after removing outliers
fig, axes = plt.subplots(1,2, figsize=(14,6))

sns.kdeplot(data=df_filtered, x='Quantity', ax=axes[0], fill=True, cut=0, bw_adjust=2)
axes[0].set_title('Distribution of Quantity')

sns.kdeplot(data=df_filtered, x='UnitPrice', ax=axes[1], fill=True, cut=0, bw_adjust=2)
axes[1].set_title('Distribution of Unit Price')

plt.tight_layout()

# **3-Feature Engineering**

In this section, I will extract features related to purchase behavior for each customer:  
 
* The most recent transaction.  
* The number of transactions.  
* The total money spent.  

I'll, then, analyze the new features.

## a) Extracting New Features

In [ ]:
df_filtered['Total Price'] = df_filtered['Quantity'] * df_filtered['UnitPrice']
df_filtered.head()

In [ ]:
date_snapshot =  df_filtered['InvoiceDate'].max() + pd.Timedelta(days=1)

customers = df_filtered.groupby('CustomerID').agg({'InvoiceDate':lambda x: (date_snapshot - x.max()).days,'InvoiceNo':'nunique','Total Price':'sum'}).rename(columns={'InvoiceDate':'Recency','InvoiceNo': 'Frequency','Total Price': 'Monetary'})

## b) Analyzing New Features (with Visualizations)

In [ ]:
print(customers.head())
print('=============================================')
print(customers.describe())

In [ ]:
# Plotting distributions
fig, axes = plt.subplots(1,3, figsize=(14,6))

sns.kdeplot(data=customers, x='Recency', ax=axes[0], fill=True, cut=0)
axes[0].set_title('Distribution of Recency')

sns.kdeplot(data=customers, x='Frequency', ax=axes[1], fill=True, cut=0)
axes[1].set_title('Distribution of Frequency')

sns.kdeplot(data=customers, x='Monetary', ax=axes[2], fill=True, cut=0)
axes[2].set_title('Distribution of Monetary')

plt.tight_layout()

# **4-Data Preprocessing**

In this section, I'll use the KMeans algorithm to segment customers.

However, the algorithm has certain assumptions that need to be satisfied. For example, the distributions are assumed to be Gaussian-like (normally distributed). The data, however, as seen above, is skewed and needs to be transformed. Additionally, KMeans is a distance-based algorithm, meaning the results would be distorted if the features are not on the same scale. It is also important to note that both frequency and monetary features have outliers, which affect certain scaling methods.

Thus, I'll do the following:  
Transform the data to be as Gaussian as possible using the PowerTransformer from scikit-learn. This transformer uses either the Yeo-Johnson (for both negative and positive values) or Box-Cox (for positive values only) methods.  
Afterwards, I'll check if there is a need to apply scaling and/or remove outliers.  

**Note**: Transforming features might resolve multiple issues at once. For example, in addition to removing skewness, features might get scaled as well.

## Transformation using PowerTransformer

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import PowerTransformer

In [ ]:
# Transform skewed features using box-cox method (since the data is only positive, box-cox is used instead for its design to deal with only positive values)
pt = PowerTransformer(method='box-cox')
customers_transformed = pt.fit_transform(customers)

In [ ]:
# I'll display the difference in distributions before and after transformation.
customers_transformed_df = pd.DataFrame(customers_transformed, index=customers.index, columns=customers.columns)

fig, axes = plt.subplots(2,3, figsize=(14,6))

sns.kdeplot(data=customers, x='Recency', ax=axes[0,0], fill=True, cut=0, color='red')
axes[0,0].set_title('Distribution of Recency (before)')
sns.kdeplot(data=customers, x='Frequency', ax=axes[0,1], fill=True, cut=0, color='red')
axes[0,1].set_title('Distribution of Frequency (before)')
sns.kdeplot(data=customers, x='Monetary', ax=axes[0,2], fill=True, cut=0, color='red')
axes[0,2].set_title('Distribution of Monetary (before)')

sns.kdeplot(data=customers_transformed_df, x='Recency', ax=axes[1,0], fill=True, cut=0, color='green')
axes[1,0].set_title('Distribution of Recency (after)')
sns.kdeplot(data=customers_transformed_df, x='Frequency', ax=axes[1,1], fill=True, cut=0, color='green')
axes[1,1].set_title('Distribution of Frequency (after)')
sns.kdeplot(data=customers_transformed_df, x='Monetary', ax=axes[1,2], fill=True, cut=0, color='green')
axes[1,2].set_title('Distribution of Monetary (after)')

plt.tight_layout()

As we can see, the transformation removed skewness and almost put all features in one scale.

# **5-Kmeans Clustering**

Now, I'll use elbow-method to find the optimal number of clusters for the data.  

**Note**: For this particuler case, it is wise to have predefined number of clusters, in this case 3, and directly use it instead of using optimization methods such elbow-method.  
The reason is that results should make sense and actionable. We might get 5 clusters as the optimized number of segments but, in reality, that number might not be interpretable, and thus hard to get the benefit out of it.

## a) Fitting KMeans

In [ ]:
# Record number of clusters and their corresponding inertia
sse = {}
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=1)
    kmeans.fit(customers_transformed_df)
    sse[k] = kmeans.inertia_ # sum of squared distances to closest cluster centroid

# Plot results
sns.pointplot(x=list(sse.keys()), y=list(sse.values()))

# Prefered point of annotation
prefered_cluster = 3
prefered_inertia = sse[3]
# I subtracted x value by 1 since pointplot of seaborn plots starting from index 0. the same goes for the other annotation
plt.annotate('Prefered Number of Clusters', xy=(prefered_cluster-1, prefered_inertia), xytext=(prefered_cluster, prefered_inertia+3000),
             arrowprops=dict(arrowstyle='->', facecolor='black'),
             fontsize=9, color='green')

# Optimized point of annotation
optimized_cluster = 4
optimized_inertia = sse[4]
plt.annotate('Optimized Number of Clusters', xy=(optimized_cluster-1, optimized_inertia), xytext=(optimized_cluster, optimized_inertia+2000),
             arrowprops=dict(arrowstyle='->', facecolor='black'),
             fontsize=9, color='red')

plt.grid(axis='y')
plt.title('Elbow Method')
plt.xlabel('k (number of clusters)'); plt.ylabel('SSE')

In [ ]:
# Fit KMeans
kmeans = KMeans(n_clusters=3, random_state=1)

## b) Assigning Clusters

In [ ]:
# Assigning
customers['Cluster'] = kmeans.fit_predict(customers_transformed_df)
customers.head()

## c) Analyzing and Assigning Clusters

In [ ]:
# Analyzing
customers.groupby('Cluster').agg({'Recency':'mean',
                                  'Frequency':'mean',
                                  'Monetary':['mean','count']}).round(0)

In [ ]:
# A distincition in segments is very clear. Hence, we can label each segment for more clarity.
cluster_mapping = {0:'Loyal',
                   1:'Casual',
                   2:'At-Risk'}

customers['Cluster'] = customers['Cluster'].map(cluster_mapping)
customers.head()

## d) Visualizing Clusters (Snake Plot)

In [ ]:
# Snake plot is a visualization to show the attributes of each cluster in one plot.
# I will use the transformed dataframe since the attributes (Recency, Frequency, and Monetary) are scaled which is mandatory for the snake plot to make sense.
# Also, the dataframe needs to be transformed into long-form in order to visualize the snake plot.
customers_transformed_df['Cluster'] = customers['Cluster']

customers_melt = pd.melt(customers_transformed_df.reset_index(),
                         id_vars=['CustomerID', 'Cluster'],
                         value_vars=['Recency', 'Frequency', 'Monetary'],
                         var_name='Attribute',
                         value_name='Value')

customers_melt.head()

In [ ]:
figure = plt.figure(figsize=(10,6))

palette = {'Loyal': 'green', 'Casual': 'grey', 'At-Risk': 'red'}

sns.pointplot(x='Attribute', y='Value', hue='Cluster', data=customers_melt, errorbar=None, hue_order=['Loyal','Casual','At-Risk'], palette=palette)

plt.grid(axis='both')
plt.title('Snake plot of scaled attributes')

# **6-Share Results**

In [ ]:
# Finally, the segments are exported as csv files and then sent to the related department

# loyal_customers = pd.Series(customers[customers['Cluster'] == 'Loyal'].index)
# loyal_customers.to_csv('loyal_customers', index=False)

# casual_customers = pd.Series(customers[customers['Cluster'] == 'Casual'].index)
# casual_customers.to_csv('casual_customers', index=False)

# at_risk_customers = pd.Series(customers[customers['Cluster'] == 'At-Risk'].index)
# at_risk_customers.to_csv('at_risk', index=False)

### **Note**: The number of customers in the original data has been reduced due to data cleaning. The customers removed can be assigned and labeled as "Casual" customers.

# **7-Future Work**

Customer segmentation can be performed and monitored periodically (monthly, quarterly, yearly, etc.). Meaning, clusters statistics (averages of recency, frequency, and monetary attributes) can be saved and compared with future statistics to evaluate marketing decisions. This approach is very advisable and effective for marketing teams.